# Practice 4: APIs & Static Web Scraping

## Topic: Public Holidays

In this practice, I will collect public holiday data using two methods:
1. REST API
2. Static web scraping with BeautifulSoup

The collected data will be processed and analyzed using Pandas.

In [1]:
import requests
import pandas as pd

In [2]:
url = "https://date.nager.at/api/v3/PublicHolidays/2026/KZ"

response = requests.get(url)

print("Status code:", response.status_code)

Status code: 200


In [3]:
data = response.json()

print(data[:3])

[{'date': '2026-01-01', 'localName': 'Жаңа жыл', 'name': "New Year's Day", 'countryCode': 'KZ', 'fixed': False, 'global': True, 'counties': None, 'launchYear': None, 'types': ['Public']}, {'date': '2026-01-02', 'localName': 'Жаңа жыл', 'name': "New Year's Day", 'countryCode': 'KZ', 'fixed': False, 'global': True, 'counties': None, 'launchYear': None, 'types': ['Public']}, {'date': '2026-03-08', 'localName': 'Халықаралық әйелдер күні', 'name': "International Women's Day", 'countryCode': 'KZ', 'fixed': False, 'global': True, 'counties': None, 'launchYear': None, 'types': ['Public']}]


In [4]:
df_api = pd.DataFrame(data)

df_api = df_api[
    ['date', 'localName', 'name', 'countryCode', 'global']
]

df_api.head()

,date,localName,name,countryCode,global
0,2026-01-01,Жаңа жыл,New Year's Day,KZ,True
1,2026-01-02,Жаңа жыл,New Year's Day,KZ,True
2,2026-03-08,Халықаралық әйелдер күні,International Women's Day,KZ,True
3,2026-03-21,Наурыз мейрамы,Nauryz Meyramy,KZ,True
4,2026-03-22,Наурыз мейрамы,Nauryz Meyramy,KZ,True


## Task 3: JSON Processing

The API data was converted into a Pandas DataFrame.
The relevant fields were selected and the columns were renamed for better readability.
Next, I will check data types, missing values, and duplicate records.

In [5]:
df_api = df_api.rename(columns={
    'date': 'Date',
    'localName': 'Local_Name',
    'name': 'Holiday_Name',
    'countryCode': 'Country_Code',
    'global': 'Global'
})

df_api.head()

,Date,Local_Name,Holiday_Name,Country_Code,Global
0,2026-01-01,Жаңа жыл,New Year's Day,KZ,True
1,2026-01-02,Жаңа жыл,New Year's Day,KZ,True
2,2026-03-08,Халықаралық әйелдер күні,International Women's Day,KZ,True
3,2026-03-21,Наурыз мейрамы,Nauryz Meyramy,KZ,True
4,2026-03-22,Наурыз мейрамы,Nauryz Meyramy,KZ,True


### Data Type Check

The data types of the API dataset are checked. The Date column will be converted to datetime format for correct date processing.

In [6]:
df_api.dtypes

,0
Date,object
Local_Name,object
Holiday_Name,object
Country_Code,object
Global,bool


In [7]:
df_api['Date'] = pd.to_datetime(df_api['Date'])

df_api.dtypes

,0
Date,datetime64[ns]
Local_Name,object
Holiday_Name,object
Country_Code,object
Global,bool


### Missing Values and Duplicates

The dataset is checked for missing values and duplicate records to ensure data quality.

In [8]:
print("Missing values:")
print(df_api.isnull().sum())

print("\nDuplicate records:", df_api.duplicated().sum())

Missing values:
Date            0
Local_Name      0
Holiday_Name    0
Country_Code    0
Global          0
dtype: int64

Duplicate records: 0


### Save API Dataset

The cleaned public holiday dataset is saved as a CSV file for further analysis.

In [10]:
df_api.to_csv('public_holidays_api.csv', index=False)

print("API dataset saved successfully.")

API dataset saved successfully.


## Task 4: Web Scraping

For the second data collection method, I will use web scraping.

The webpage contains information about holidays in Kazakhstan in 2026.
I will use Requests and BeautifulSoup to extract holiday data from the HTML page.

Webpage:
https://www.timeanddate.com/holidays/kazakhstan/2026

In [11]:
from bs4 import BeautifulSoup
import requests
import pandas as pd

In [13]:
web_url = "https://www.officeholidays.com/countries/kazakhstan"

headers = {
    "User-Agent": "Mozilla/5.0"
}

web_response = requests.get(web_url, headers=headers)

print("Status code:", web_response.status_code)


Status code: 200


### HTML Parsing

The webpage was successfully accessed with status code 200.
BeautifulSoup is used to parse the HTML content and find the holiday table.

In [14]:
soup = BeautifulSoup(web_response.text, "html.parser")

table = soup.find("table")

print(table is not None)

True


In [15]:
rows = table.find_all("tr")

print("Number of rows:", len(rows))

for row in rows[:5]:
    cells = row.find_all(["th", "td"])
    print([cell.get_text(strip=True) for cell in cells])

Number of rows: 18
['Day', 'Date', 'Holiday Name', 'Type', 'Comments']
['Thursday', 'Jan 01', "New Year's Day", 'National Holiday', '']
['Friday', 'Jan 02', "Day after New Year's Day", 'National Holiday', '']
['Wednesday', 'Jan 07', 'Orthodox Christmas Day', 'National Holiday', '']
['Monday', 'Mar 09', "Women's Day (in lieu)", 'National Holiday', '']


### Extract Holiday Data

The holiday records are extracted from the HTML table. Five fields are collected: Day, Date, Holiday Name, Type, and Comments. The extracted data is stored in a Pandas DataFrame.

In [16]:
holidays = []

for row in rows[1:]:
    cells = row.find_all("td")

    if len(cells) >= 5:
        holidays.append({
            "Day": cells[0].get_text(strip=True),
            "Date": cells[1].get_text(strip=True),
            "Holiday_Name": cells[2].get_text(strip=True),
            "Type": cells[3].get_text(strip=True),
            "Comments": cells[4].get_text(strip=True)
        })

df_web = pd.DataFrame(holidays)

df_web.head()

,Day,Date,Holiday_Name,Type,Comments
0,Thursday,Jan 01,New Year's Day,National Holiday,
1,Friday,Jan 02,Day after New Year's Day,National Holiday,
2,Wednesday,Jan 07,Orthodox Christmas Day,National Holiday,
3,Monday,Mar 09,Women's Day (in lieu),National Holiday,
4,Sunday,Mar 15,Constitution Day (in lieu),National Holiday,


### Web Data Quality Check

The scraped dataset is checked for empty values and duplicate records to evaluate its data quality.

In [17]:
print("Dataset shape:", df_web.shape)

print("\nMissing values:")
print(df_web.isnull().sum())

print("\nEmpty values:")
print((df_web == "").sum())

print("\nDuplicate records:", df_web.duplicated().sum())

Dataset shape: (17, 5)

Missing values:
Day             0
Date            0
Holiday_Name    0
Type            0
Comments        0
dtype: int64

Empty values:
Day              0
Date             0
Holiday_Name     0
Type             0
Comments        12
dtype: int64

Duplicate records: 0


### Save Scraped Dataset

The scraped holiday dataset is saved as a CSV file for further review and comparison.

In [18]:
df_web.to_csv("public_holidays_web.csv", index=False)

print("Web dataset saved successfully.")

Web dataset saved successfully.


## Task 5: Data Review

The API and web scraping datasets are reviewed and compared.
Their size, columns, missing values, duplicate records, and data types are checked.

In [19]:
print("=== API DATASET ===")
print("Shape:", df_api.shape)
print("Columns:", df_api.columns.tolist())
print("\nMissing values:")
print(df_api.isnull().sum())
print("\nDuplicates:", df_api.duplicated().sum())
print("\nData types:")
print(df_api.dtypes)

print("\n============================")

print("\n=== WEB SCRAPING DATASET ===")
print("Shape:", df_web.shape)
print("Columns:", df_web.columns.tolist())
print("\nMissing values:")
print(df_web.isnull().sum())
print("\nEmpty values:")
print((df_web == "").sum())
print("\nDuplicates:", df_web.duplicated().sum())
print("\nData types:")
print(df_web.dtypes)

=== API DATASET ===
Shape: (13, 5)
Columns: ['Date', 'Local_Name', 'Holiday_Name', 'Country_Code', 'Global']

Missing values:
Date            0
Local_Name      0
Holiday_Name    0
Country_Code    0
Global          0
dtype: int64

Duplicates: 0

Data types:
Date            datetime64[ns]
Local_Name              object
Holiday_Name            object
Country_Code            object
Global                    bool
dtype: object


=== WEB SCRAPING DATASET ===
Shape: (17, 5)
Columns: ['Day', 'Date', 'Holiday_Name', 'Type', 'Comments']

Missing values:
Day             0
Date            0
Holiday_Name    0
Type            0
Comments        0
dtype: int64

Empty values:
Day              0
Date             0
Holiday_Name     0
Type             0
Comments        12
dtype: int64

Duplicates: 0

Data types:
Day             object
Date            object
Holiday_Name    object
Type            object
Comments        object
dtype: object


### Findings

1. The API dataset contains 13 records and 5 columns, while the web scraping dataset contains 17 records and 5 columns.

2. The API dataset has no missing values or duplicate records, which shows that the collected API data is clean.

3. The web scraping dataset also has no duplicate records. However, the Comments column contains 12 empty values because comments are not provided for every holiday.

4. The API provides structured fields such as local holiday name, country code, and global status. The webpage provides different information, such as day of the week, holiday type, and comments.

5. The Date column in the API dataset was converted to datetime format. In the scraped dataset, the Date column is stored as an object because the website provides dates in text format such as "Jan 01".

### API vs Web Scraping

The API method was easier to process because the data was already provided in structured JSON format. It was simple to select fields and convert them into a Pandas DataFrame.

Web scraping required more steps because the HTML structure had to be parsed with BeautifulSoup. The correct table, rows, and cells had to be identified before creating the DataFrame.

Overall, the API was more convenient for structured data collection, while web scraping was useful for collecting information directly from a webpage.